In [1]:
!nvidia-smi

Fri May  8 17:27:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
%%writefile cuda_full_input.cu
#include <stdio.h>
__global__ void matrixMul(int *A, int *B, int *C, int n) {
int row = blockIdx.y * blockDim.y + threadIdx.y;
int col = blockIdx.x * blockDim.x + threadIdx.x;
if (row < n && col < n) {
int sum = 0;
for (int k = 0; k < n; k++) {
sum += A[row * n + k] * B[k * n + col];
}
C[row * n + col] = sum;
}
}

int main() {
int m;
printf("Enter size of square matrix: ");
scanf("%d", &m);
int matrixSize = m * m * sizeof(int);
int h_MA[m*m], h_MB[m*m], h_MC[m*m];
printf("Enter elements of Matrix A:\n");
for(int i = 0; i < m*m; i++)
scanf("%d", &h_MA[i]);
printf("Enter elements of Matrix B:\n");
for(int i = 0; i < m*m; i++)
scanf("%d", &h_MB[i]);

int *d_MA, *d_MB, *d_MC;
cudaMalloc((void**)&d_MA, matrixSize);
cudaMalloc((void**)&d_MB, matrixSize);
cudaMalloc((void**)&d_MC, matrixSize);
cudaMemcpy(d_MA, h_MA, matrixSize, cudaMemcpyHostToDevice);
cudaMemcpy(d_MB, h_MB, matrixSize, cudaMemcpyHostToDevice);
dim3 threadsPerBlock(16,16);
dim3 blocksPerGrid((m+15)/16,(m+15)/16);
matrixMul<<<blocksPerGrid, threadsPerBlock>>>(d_MA, d_MB, d_MC, m);
cudaMemcpy(h_MC, d_MC, matrixSize, cudaMemcpyDeviceToHost);
printf("Matrix Multiplication Result:\n");
for(int i = 0; i < m; i++) {
for(int j = 0; j < m; j++) {
printf("%d ", h_MC[i*m + j]);
}
printf("\n");
}
return 0;
}

Overwriting cuda_full_input.cu


In [7]:
!nvcc cuda_full_input.cu -o run

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [12]:
!./run

Enter size of square matrix: 2
Enter elements of Matrix A:

^C
